In [902]:
# Alejandro J Rivera

In [903]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [904]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
# Name each column
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']

# Read the data from the url
raw_dataset = pd.read_csv(url, names=column_names,
                          na_values='?', comment='\t',
                          sep=' ', skipinitialspace=True)

dataset = raw_dataset.copy()

# Removes missing data
dataset = dataset.dropna()

dataset['Origin'] = dataset['Origin'].map({1: 'USA', 2: 'Europe', 3: 'Japan'})

dataset = pd.get_dummies(dataset, columns=['Origin'], prefix='', prefix_sep='', dtype=float)

In [905]:
# Takes the data that will be used for training and the data that will be used for testing
train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)

train_dataset.describe().transpose()

# Copies the data from the dataset that will be used as features
train_features = train_dataset.copy()
test_features = test_dataset.copy()

# Divides the data that will be used as labels from the features
train_labels = train_features.pop('MPG')
test_labels = test_features.pop('MPG')

train_dataset.describe().transpose()[['mean', 'std']]

,mean,std
MPG,23.310510,7.728652
Cylinders,5.477707,1.699788
Displacement,195.318471,104.331589
Horsepower,104.869427,38.096214
Weight,2990.251592,843.898596
Acceleration,15.559236,2.789230
Model Year,75.898089,3.675642
Europe,0.178344,0.383413
Japan,0.197452,0.398712
USA,0.624204,0.485101


In [906]:
# Calculates the mean and std from the features
features_mean = train_features.mean()
features_std = train_features.std()

# Normalize the features
train_features = (train_features - features_mean) / features_std
test_features = (test_features - features_mean) / features_std

# Turn the features into tensors
train_features = torch.tensor(train_features.values, dtype=torch.float32)
test_features = torch.tensor(test_features.values, dtype=torch.float32)

# Calculates the mean and std from the labels
labels_mean = train_labels.mean()
labels_std = train_labels.std()

# Normalize the labels
train_labels = (train_labels - labels_mean) / labels_std
test_labels = (test_labels - labels_mean) / labels_std

# Turn the labels into tensors
train_labels = torch.tensor(train_labels.values.reshape(-1, 1), dtype=torch.float32)
test_labels = torch.tensor(test_labels.values.reshape(-1, 1), dtype=torch.float32)

# Amount of features that will be used in the first layer
features_size = train_features.shape[1]

In [907]:
# Basic regression neural network 
class NeuralNet1(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(features_size, 1)
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits
   
# 4 layer deep neural network
class NeuralNet2(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(features_size, 10),
            nn.ReLU(),
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits
    
# 5 layer deep neural network
class NeuralNet3(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(features_size, 10),
            nn.ReLU(),
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 30),
            nn.ReLU(),
            nn.Linear(30, 20),
            nn.ReLU(),
            nn.Linear(20, 1),
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits

In [908]:
networks = [NeuralNet1, NeuralNet2, NeuralNet3]
errors = []

# Number of epochs each model will be trained for
epochs = 20

# Takes each neural network and trains it for the number of epochs above with a learning rate of 0.01
for network in networks:
    model = network()
    
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.01)

    for epoch in range(epochs):
        optimizer.zero_grad()

        outputs = model(train_features)
        loss = criterion(outputs, train_labels)

        loss.backward()
        optimizer.step()

    val_outputs = model(test_features)
    error = criterion(val_outputs, test_labels)

    errors.append(error)

network1 = errors[0]
network2 = errors[1]
network3 = errors[2]

In [909]:
print("Amount of error for validation:")
print(f"Neural Network 1: {network1}")
print(f"Neural Network 2: {network2}")
print(f"Neural Network 3: {network3}")

Amount of error for validation:
Neural Network 1: 0.33593446016311646
Neural Network 2: 0.24815933406352997
Neural Network 3: 0.1766795516014099


In [910]:
# Determines which neural network had the least amount of error for validation
if network1 < network2 and network1 < network3:
    print(f"The model with least amount of error for validation is model 1, with a validation error of {network1}")
elif network2 < network1 and network2 < network3:
    print(f"The model with least amount of error for validation is model 2, with a validation error of {network2}")
else:
    print(f"The model with least amount of error for validation is model 3, with a validation error of {network3}")

The model with least amount of error for validation is model 3, with a validation error of 0.1766795516014099
